# Attempt 2 with Langchain only

In [29]:
from os import environ
from dotenv import load_dotenv
load_dotenv()

True

In [30]:
from langchain_openai import OpenAIEmbeddings

embeddings_client = OpenAIEmbeddings(
    base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model=environ.get("GITHUB_EMBEDDINGS_MODEL_ID")  # 🎯 Selected AI model
)

In [31]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="LLM_Powered_Autonomous_Agents",
    embedding_function=embeddings_client,
    persist_directory="./.chroma_db"
)

In [32]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
# A special class/object that filters HTML documents for relevant data
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
# A "Loader" is an object representing the method for grabbing data, in this case, from a public website with sample data
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [33]:
# Print out first 500 characters from the document
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [34]:
# TODO: Find a semantic method of splitting a document via a "change in context", i.e. a topic change

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [35]:
# Shoves our document chunks into the embeddings model, and stores them in our local chroma vector store database
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['fea053ed-729e-433f-ac64-2702e9d36b13', '7da2e284-f895-41ef-9d5a-51d8a92ca789', 'd7e310c0-41cf-4f3c-af2f-8fb8202011ba']


In [36]:
# Defines a simple RAG tool function for an agent to use
from langchain.tools import tool

# The "response_format=" argument is used when a tool function returns 2 values:
# 1. The string message result to send to the model
# 2. An "artifact" to couple with the tool's result, in this case, the actual retrieved document objects themselves
# In this case, this allows us (not necessarily the LLM in the agent) to access the document's metadata more easily when we interact with the RAG tool
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""

    # K-nearest neighbours search with Chroma
    retrieved_docs = vector_store.similarity_search(query, k=2)

    # The "serialised" result, which is just a long string combining documents (chunks) with their metadata
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs

# Extra note: you can easily ask the calling LLM to provide more arguments to a tool function:
# from typing import Literal
# def retrieve_context(query: str, section: Literal["beginning", "middle", "end"]):

In [37]:
from langchain_openai import ChatOpenAI

chat_client = ChatOpenAI(
    base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model= environ.get("GITHUB_MODEL_ID")  # 🎯 Selected AI model
)

In [38]:
from langchain.agents import create_agent

agent = create_agent(
    model=chat_client,
    tools=[retrieve_context],
    system_prompt=(
        "You have access to a tool that retrieves context from a blog post. ",
        "Use the tool to help answer user queries.")
)

In [45]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

# agent.invoke(query)

# agent.invoke(
#     {"input": [{"role": "user", "content": "What is the standard method for Task Decomposition?\n\nOnce you get the answer, look up common extensions of that method."}]}
# )

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.


BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'messages[0].content[0]': expected an object, but got a string instead.", 'type': 'invalid_request_error', 'param': 'messages[0].content[0]', 'code': 'invalid_type'}}